In [1]:
import pandas as pd
import numpy as np
#Load Data
patients_df = pd.read_csv("patient_data.csv")
billing_df = pd.read_csv("billing_data.csv")
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [2]:
patients_df = patients_df.drop(
    columns=['ReceptionistID', 'CheckInTime'],
    errors='ignore'
)
patients_df = patients_df[
    ['PatientID', 'Department', 'Doctor', 'BillAmount']
]
patients_df = patients_df.drop_duplicates(
    subset='PatientID'
)

patients_df['BillAmount'] = patients_df['BillAmount'].fillna(
    patients_df['BillAmount'].mean()
)

In [3]:
#Analysis
dept_bill = patients_df.groupby('Department')['BillAmount'].sum()
print(dept_bill)


Department
Cardiology     11200.000000
Dermatology     6233.333333
Neurology       6233.333333
Orthopedics     7500.000000
Name: BillAmount, dtype: float64


In [4]:
#Merge
merged_df = pd.merge(patients_df, billing_df, on='PatientID', how='inner')
print("\n--- Merged Data ---")
print(merged_df.head())


--- Merged Data ---
   PatientID   Department     Doctor   BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000              2000   
1        102    Neurology   Dr. John  6233.333333              1500   
2        103  Orthopedics    Dr. Lee  7500.000000              2500   
3        104   Cardiology  Dr. Smith  6200.000000              3000   
4        105  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [5]:
#Add new data
new_patients = pd.DataFrame({
    'PatientID': [101, 102],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr.A', 'Dr.B'],
    'BillAmount': [5000, 7000]
})

In [6]:
# Align columns before concatenation
for col in merged_df.columns:
    if col not in new_patients.columns:
        new_patients[col] = np.nan

new_patients = new_patients[merged_df.columns]

# Row-wise concatenation
merged_df = pd.concat(
    [merged_df, new_patients],
    ignore_index=True
)

In [7]:
# Column-wise concatenation
extra_columns = pd.DataFrame({
    'InsuranceCovered': [True] * len(merged_df),
    'DiscountPercent': [10] * len(merged_df)
})

merged_df = pd.concat(
    [merged_df, extra_columns],
    axis=1
)

In [8]:
# Final amount calculation
merged_df['FinalAmount'] = (
    merged_df['BillAmount']
    - (merged_df['BillAmount']
       * merged_df['DiscountPercent'] / 100)
)

In [9]:
# Output
print("\n--- Final Dataset ---")
print(merged_df.head())

print("\nDataset Shape:", merged_df.shape)


--- Final Dataset ---
   PatientID   Department     Doctor   BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000            2000.0   
1        102    Neurology   Dr. John  6233.333333            1500.0   
2        103  Orthopedics    Dr. Lee  7500.000000            2500.0   
3        104   Cardiology  Dr. Smith  6200.000000            3000.0   
4        105  Dermatology   Dr. Rose  6233.333333            1000.0   

   FinalAmount  InsuranceCovered  DiscountPercent  
0       4500.0              True               10  
1       5610.0              True               10  
2       6750.0              True               10  
3       5580.0              True               10  
4       5610.0              True               10  

Dataset Shape: (7, 8)
